In [1]:
from ingest import load_faq_data

documents = load_faq_data()
documents = [doc for doc in documents if doc["course"] == "llm-zoomcamp"]
len(documents)

113

In [2]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [3]:
from pydantic import BaseModel


class Questions(BaseModel):
    questions: list[str]

In [4]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [5]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [6]:
import json

from evaluation_utils import llm_structured

user_prompt = json.dumps(doc)
result, usage = llm_structured(openai_client, data_gen_instructions, user_prompt, Questions)
print(result.questions)

['I just found this course — is it still okay to join now?', 'Can I join late, or is it too late to start?', 'If I start the course now, can I still get a certificate?', 'What do I need to do to be eligible for the certificate if I join late?', 'Is there still time to submit the project for the certificate?']


In [7]:
from evaluation_utils import calc_price

calc_price(usage)

{'input_cost': 0.00015525,
 'output_cost': 0.00038700000000000003,
 'total_cost': 0.00054225}

In [8]:
from evaluation_utils import llm_structured_retry


def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions,
    )

    results = []
    for q in out.questions:
        results.append({"question": q, "document": doc["id"]})

    return results, usage

In [9]:
from concurrent.futures import ThreadPoolExecutor

from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/113 [00:00<?, ?it/s]

In [10]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

565

In [11]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.08922299999999996

In [12]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)
df_ground_truth.head()

,question,document
0,I just found this course late — can I still jo...,74eb249bbf
1,"If I enroll after the course started, am I sti...",74eb249bbf
2,What do I need to do to qualify for a certific...,74eb249bbf
3,Is there a deadline for project submission if ...,74eb249bbf
4,"Can I still take the course now, and will a la...",74eb249bbf
